This book aims to achive a set of hyperparameters for XGBoost which give best results and then use them to make oof predictions.

In [1]:
import pandas as pd

path = '/kaggle/input/competitions/playground-series-s6e5/'

#train data
train_df = pd.read_csv(path + 'train.csv')
print(train_df.shape)
display(train_df.head())

print()

#test data:
test_df = pd.read_csv(path + 'test.csv')
print(test_df.shape)
display(test_df.head())

(439140, 16)


,id,Driver,Compound,Race,Year,PitStop,LapNumber,Stint,TyreLife,Position,LapTime (s),LapTime_Delta,Cumulative_Degradation,RaceProgress,Position_Change,PitNextLap
0,0,D109,HARD,Canadian Grand Prix,2022,0,50,2,39.0,8,78.491,-7.564,21.019,0.714286,5.0,1.0
1,1,D086,HARD,Dutch Grand Prix,2025,1,27,2,7.0,4,75.095,-32.617,-223.207,0.346154,-3.0,0.0
2,2,ZON,HARD,Austrian Grand Prix,2022,0,59,3,22.0,13,70.945,-7.540,-100.529,0.819444,3.0,1.0
3,3,SPE,MEDIUM,Pre-Season Testing,2023,0,2,1,2.0,7,94.361,-7.324,-7.324,0.076923,0.0,0.0
4,4,D019,HARD,Azerbaijan Grand Prix,2022,1,26,3,6.0,2,107.878,8.965,-14.139,0.361111,3.0,0.0



(188165, 15)


,id,Driver,Compound,Race,Year,PitStop,LapNumber,Stint,TyreLife,Position,LapTime (s),LapTime_Delta,Cumulative_Degradation,RaceProgress,Position_Change
0,439140,D119,MEDIUM,British Grand Prix,2023,0,21,1,21.0,4,93.387,0.280,-4.984,0.403846,0.0
1,439141,VER,MEDIUM,Abu Dhabi Grand Prix,2023,0,24,1,24.0,1,90.867,-0.129,-1.990,0.413793,0.0
2,439142,D270,MEDIUM,British Grand Prix,2023,0,24,1,24.0,11,92.871,0.041,-8.842,0.461538,0.0
3,439143,D112,SOFT,São Paulo Grand Prix,2024,0,6,2,4.0,15,94.967,-19.741,8.250,0.077922,1.0
4,439144,AND,HARD,United States Grand Prix,2024,0,52,2,29.0,12,99.112,0.930,-20.848,0.722222,7.0


In [2]:
target = 'PitNextLap'

X = train_df.drop(columns=['id',target], axis=1)
y = train_df[target]

X_test = test_df.drop('id', axis=1)

converting all the categorical faetures dtype from object --> category, to enable categorical handling for xgboost

In [3]:
cat_cols = list(X.select_dtypes('object').columns)
num_cols = list(X.select_dtypes(exclude=['object']).columns)

for c in cat_cols:
    X[c]       =   X[c].astype('category')
    X_test[c]  =   X_test[c].astype('category')

## GroupKFold
GroupKFold ensures the model is evaluated on completely unseen races, not on laps sandwiched between training laps from the same race, which would leak sequential information and inflate AUC artificially.



In [4]:
#optuna tuned
xgb_params = {
    "learning_rate": 0.005152269004557939, 
    "max_depth": 10, 
    "min_child_weight": 11.83503440474689, 
    "subsample": 0.5144055150851956, 
    "gamma": 0.17091899550489928,
    "colsample_bytree": 0.5059508549225745, 
    "colsample_bylevel": 0.9945701925466703, 
    "reg_lambda": 0.032776579651102616, 
    "reg_alpha": 0.0155251922035826,
    "max_delta_step": 6.937126157478375,
                      
    "n_estimators": 30_000,
    "enable_categorical": True,
    "objective": "binary:logistic",
    "eval_metric": "auc",
    "device": "cuda",
    "early_stopping_rounds": 500,
    "random_state": 42,
}

In [5]:
from sklearn.model_selection import GroupKFold
from xgboost import XGBClassifier
from sklearn.metrics import roc_auc_score
import numpy as np

groups = train_df['Race'].astype(str) + '_' + train_df['Year'].astype(str)

gkf = GroupKFold(n_splits=5)

oof_probs  = np.zeros(len(X))
test_probs = np.zeros(len(X_test))  # ← correct length
auc_scores = []

for fold, (train_idx, val_idx) in enumerate(gkf.split(X, y, groups)):
    print(f"Fold {fold+1}...")
    X_tr, X_val = X.iloc[train_idx], X.iloc[val_idx]
    y_tr, y_val = y.iloc[train_idx], y.iloc[val_idx]

    model = XGBClassifier(**xgb_params)
    
    model.fit(
        X_tr, y_tr,
        eval_set=[(X_val, y_val)],
        verbose=False
    )

    oof_probs[val_idx] = model.predict_proba(X_val)[:, 1]
    test_probs += model.predict_proba(X_test)[:, 1] / gkf.n_splits  

    auc = roc_auc_score(y_val, oof_probs[val_idx])
    auc_scores.append(auc)
    print(f"  Fold {fold+1} AUC: {auc:.5f}")

print(f"\nMean OOF AUC : {roc_auc_score(y, oof_probs):.5f}")  
print(f"Mean Fold AUC: {np.mean(auc_scores):.5f}")

Fold 1...


/usr/local/lib/python3.12/dist-packages/xgboost/core.py:751: UserWarning: [16:36:19] WARNING: /__w/xgboost/xgboost/src/common/error_msg.cc:62: Falling back to prediction using DMatrix due to mismatched devices. This might lead to higher memory usage and slower performance. XGBoost is running on: cuda:0, while the input data is on: cpu.
Potential solutions:
- Use a data structure that matches the device ordinal in the booster.
- Set the device for booster before call to inplace_predict.

This warning will only be shown once.

  return func(**kwargs)


  Fold 1 AUC: 0.93224
Fold 2...
  Fold 2 AUC: 0.92066
Fold 3...
  Fold 3 AUC: 0.94115
Fold 4...
  Fold 4 AUC: 0.93206
Fold 5...
  Fold 5 AUC: 0.92114

Mean OOF AUC : 0.92978
Mean Fold AUC: 0.92945


In [6]:
submission = pd.DataFrame({
    "id": test_df["id"],
    target: test_probs
})

submission.to_csv("submission.csv", index=False)